# Streaming Reservoir Computing From TDMS States

This notebook trains the readout directly from measured TDMS reservoir states.
The modification follows the `RC_MGV1.ipynb` readout idea: use a fixed sampled
camera/state feature vector and train the output by ridge regression after a
washout period. The TDMS frames are still streamed, so the full state matrix is
not kept in memory.


In [ ]:
from pathlib import Path
import glob
import numpy as np
import matplotlib.pyplot as plt

try:
    from nptdms import TdmsFile
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "nptdms is not installed. Install it first with: python3 -m pip install nptdms"
    ) from exc


In [ ]:
# =========================
# Configuration
# =========================
# TDMS can be a single .tdms file, a glob, or a directory containing .tdms files.
TDMS_PATHS = Path("/mnt/71/data/JINGYE/170826")

GROUP = "State"
CHANNEL = "stateu16"
RAW_FRAME_SHAPE = (1024, 1024)
BLOCK_SHAPE = (4, 4)          # 1024x1024 -> 256x256
BLOCK_REDUCER = "mean"

# Each Template_###_video.tdms contributes exactly one state image.
# 0 means first frame in each TDMS; use 4 if the fifth frame is the stable measured state.
SELECTED_FRAME_INDEX = 0

TDMS_START_INDEX = 1
TDMS_STOP_INDEX = 2964
N_SAMPLES = TDMS_STOP_INDEX - TDMS_START_INDEX + 1
N_TRAIN = 2000
WASHOUT = 200

# One-step-ahead task: state(t) predicts truth(t+1).
# Train stage runs states 1-2000. The first 200 are washout and do not train W_out.
# Ridge fit uses states 201-2000 -> truth labels 202-2001.
# Test stage: after W_out is fixed, states 2001-2964 make predictions for truth labels 2002-2965.
TRAIN_STATE_START = 1
TRAIN_STATE_STOP = N_TRAIN
TEST_STATE_START = N_TRAIN + 1
TEST_STATE_STOP = N_SAMPLES
N_TRAIN_PAIRS = TRAIN_STATE_STOP - TRAIN_STATE_START + 1
N_TEST_PAIRS = TEST_STATE_STOP - TEST_STATE_START + 1

MG_ARRAYS_PATH = Path("jy RC") / "Exp August" / "measurement_mg_arrays_t3000_tau17.npz"
RESULT_PATH = Path("streaming_tdms_rc_results_ridge_256x256.npz")

# Readout settings ported from RC_MGV1: use a fixed sampled camera/state vector,
# then solve Wout with ridge regression after the washout part of the train stage.
READOUT_FEATURE_MODE = "raw"       # "raw", "sqrt_intensity", or "binary_threshold"
N_READOUT_FEATURES = 1024          # fixed sampled features from 256x256; do not set None unless memory is enough
RANDOM_SEED = 42
RIDGE_ALPHA = 1e-4
SOLVE_JITTER = 1e-6
ADD_BIAS = False             # match RC_MGV1: y = X @ W_out, no explicit bias column
PRINT_EVERY = 100
